In [1]:
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)

import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
datapath = '../data/2010-2011 Solar home electricity data.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_01"

In [2]:
datapath = '../data/2011-2012 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_02"

shape: (270_304, 54)
┌──────────┬───────────┬──────────┬──────────────────────┬───┬───────┬───────┬───────┬─────────────┐
│ Customer ┆ Generator ┆ Postcode ┆ Consumption Category ┆ … ┆ 23:00 ┆ 23:30 ┆ 0:00  ┆ Row Quality │
│ ---      ┆ Capacity  ┆ ---      ┆ ---                  ┆   ┆ ---   ┆ ---   ┆ ---   ┆ ---         │
│ i64      ┆ ---       ┆ i64      ┆ str                  ┆   ┆ f64   ┆ f64   ┆ f64   ┆ str         │
│          ┆ f64       ┆          ┆                      ┆   ┆       ┆       ┆       ┆             │
╞══════════╪═══════════╪══════════╪══════════════════════╪═══╪═══════╪═══════╪═══════╪═════════════╡
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 1.063 ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ GC                   ┆ … ┆ 0.118 ┆ 0.219 ┆ 0.162 ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ GG                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 0.0   ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆

In [ ]:
datapath = '../data/2012-2013 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_03"

In [3]:
# we can get the training and testing customers from the csv file
training_customers = np.loadtxt('../data/training_customers.csv', dtype=int)
testing_customers = np.loadtxt('../data/testing_customers.csv', dtype=int)

In [ ]:
# alternatively, get all the unique customers as their own dataframes
customers = df['Customer'].unique()
# pick 80% of the random customers as training data
training_customers = np.random.choice(customers, int(0.8*len(customers)), replace=False)
# the rest of the customers are testing data
testing_customers = np.setdiff1d(customers, training_customers)

In [ ]:
# save the customers number to a csv file
np.savetxt('../data/training_customers.csv', training_customers, fmt='%s')
np.savetxt('../data/testing_customers.csv', testing_customers, fmt='%s')

In [4]:
from helper import transform_polars_df
# loop through each customer and use transform_polars_df to get the dataframe and store it in a list call dataset
training_dataset = []
for customer in training_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as training dataset: {customer}")
        print(e)
        break
    training_dataset.append(newcustomerdf)

testing_dataset = []
for customer in testing_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as testing dataset: {customer}")
        print(e)
        break
    testing_dataset.append(newcustomerdf)

In [ ]:
# provide std and mean on the different columns of the training dataset
testdf = testing_dataset[25]
# drop timestamp and time columns
testdf = testdf.drop(['Timestamp', 'Time'])
print(testdf.describe())

In [5]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env

testing_env_fns = [make_env(ds) for ds in testing_dataset]

num_step = None # pick the number of step for the simulation/none for full length
test_envs = [env_fn(num_step) for env_fn in testing_env_fns]

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [15]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env
# Create a list of environment creation functions to build a vectorized environment.
training_env_fns = [make_env(ds) for ds in training_dataset]
#training_vec_env = DummyVecEnv(training_env_fns)
num_step = None # pick the number of step for the simulation/none for full length
train_envs = [env_fn(num_step) for env_fn in training_env_fns]

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [ ]:
# combine the test_envs and train_envs into a single list
combined_envs = test_envs + train_envs

In [16]:
selected_list = train_envs
if selected_list is test_envs:
    env_type = "test"
    env_fns = testing_env_fns
elif selected_list is train_envs:
    env_type = "train"
    env_fns = training_env_fns
else:
    env_type = "combined"
    env_fns = testing_env_fns + training_env_fns

In [ ]:
from decision import Agent, run_episodes_parallel
rule_agent_kwargs = {
    'algorithm': 'rule'
}

# run episodes in the list in parallel using the rule-based agent on the training environments
episode_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=rule_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

In [ ]:
# Collect all schemas
schemas = [df.schema for df in dfs_with_id]

# Find the most common schema (assume it's the correct one)
from collections import Counter
schema_counts = Counter([tuple(sorted(s.items())) for s in schemas])
most_common_schema = dict(schema_counts.most_common(1)[0][0])

# Print out indices and details of DataFrames with mismatched schemas
for i, schema in enumerate(schemas):
    if dict(sorted(schema.items())) != most_common_schema:
        print(f"DF {i} schema mismatch:")
        print("Schema:", schema)
        print("Difference:", set(schema.items()) ^ set(most_common_schema.items()))

In [ ]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_logs)]
rule_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/rule_{env_type}_{episode_num}_logs.parquet"
rule_all_logs.write_parquet(file_name)

In [7]:
from decision import Agent, run_episodes_parallel, run_single
# Initialize environments and SDP agent parameters
sdp_agent_kwargs = {
    'algorithm': 'sdp',
    'soc_resolution': 20,
    'action_resolution': 41,  # best to be 2*soc_resolution + 1
    'degradation_model': 'static' # the other option being static degradation 'static'
    #'linear_deg_cost_p_kwh': 0.2
}

# Run a single episode for timing test
#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=sdp_agent_kwargs, render=False, display_progress=True)


# Run all episodes in parallel
sdp_episode_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=sdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

[INFO] Starting 60 episodes with max_workers=12


Episodes:  50%|█████     | 30/60 [1:36:23<4:11:50, 503.69s/it]

[START] Episode 6
Sim Complete
[DONE]  Episode 6 (Elapsed: 1988.88 sec)
[START] Episode 13
Sim Complete
[DONE]  Episode 13 (Elapsed: 1982.88 sec)
[START] Episode 25


Episodes:  52%|█████▏    | 31/60 [1:38:38<3:09:59, 393.10s/it]

[START] Episode 7
Sim Complete
[DONE]  Episode 7 (Elapsed: 1986.52 sec)
[START] Episode 12
Sim Complete
[DONE]  Episode 12 (Elapsed: 26.05 sec)
[START] Episode 15
Sim Complete
[DONE]  Episode 15 (Elapsed: 1982.50 sec)
[START] Episode 26
Sim Complete
[DONE]  Episode 26 (Elapsed: 4.37 sec)
[START] Episode 27


...

[START] Episode 8
Sim Complete
[DONE]  Episode 8 (Elapsed: 2019.08 sec)
[START] Episode 19
Sim Complete
[DONE]  Episode 19 (Elapsed: 2005.15 sec)
[START] Episode 31
Sim Complete
[DONE]  Episode 31 (Elapsed: 1756.31 sec)
[START] Episode 41


Episodes:  98%|█████████▊| 59/60 [2:36:22<04:07, 247.50s/it]  

Sim Complete
[DONE]  Episode 27 (Elapsed: 1935.65 sec)
[START] Episode 43
Sim Complete
[DONE]  Episode 43 (Elapsed: 1919.97 sec)
[START] Episode 57


Episodes: 100%|██████████| 60/60 [2:37:28<00:00, 157.48s/it]


Sim Complete
[DONE]  Episode 29 (Elapsed: 1959.15 sec)
[START] Episode 45
Sim Complete
[DONE]  Episode 45 (Elapsed: 1943.08 sec)
Sim Complete
[DONE]  Episode 28 (Elapsed: 1942.11 sec)
[START] Episode 44
Sim Complete
[DONE]  Episode 44 (Elapsed: 1937.81 sec)
[START] Episode 58
Sim Complete
[DONE]  Episode 58 (Elapsed: 31.40 sec)
Sim Complete
[DONE]  Episode 32 (Elapsed: 1962.82 sec)
[START] Episode 47
Sim Complete
[DONE]  Episode 47 (Elapsed: 1944.76 sec)
Sim Complete
[DONE]  Episode 30 (Elapsed: 1956.85 sec)
[START] Episode 46
Sim Complete
[DONE]  Episode 46 (Elapsed: 63.75 sec)
[START] Episode 50
Sim Complete
[DONE]  Episode 50 (Elapsed: 1938.97 sec)
Sim Complete
[DONE]  Episode 36 (Elapsed: 1967.60 sec)
[START] Episode 49
Sim Complete
[DONE]  Episode 49 (Elapsed: 1960.22 sec)
Sim Complete
[DONE]  Episode 38 (Elapsed: 1971.88 sec)
[START] Episode 51
Sim Complete
[DONE]  Episode 51 (Elapsed: 1951.55 sec)
Sim Complete
[DONE]  Episode 39 (Elapsed: 1959.49 sec)
[START] Episode 52
Sim Comp

In [8]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(sdp_episode_logs)]
sdp_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/sdp_{env_type}_{episode_num}_logs.parquet"
sdp_all_logs.write_parquet(file_name)

In [9]:
from decision import Agent, run_episodes_parallel, run_single
mrdp_agent_kwargs = {
    'algorithm': 'mrdp',
    'soc_resolution': 20,           # fallback/default for single-horizon
    'action_resolution': 41,        # fallback/default for single-horizon
    'degradation_model': 'static',  # or 'static'
    #'linear_deg_cost_p_kwh': 0.2,   # only needed if using linear
    'subhorizon_specs': [
        {
            'start': 0,
            'length': 12,           # e.g. 6 hours at 30-min steps
            'soc_res': 20,          # fine SoC discretization
            'action_res': 11,       # fine action discretization
            'step_duration': 0.5    # hours per step (30 min)
        },
        {
            'start': 12,
            'length': 36,           # e.g. 18 hours at 30-min steps
            'soc_res': 8,           # coarse SoC discretization
            'action_res': 5,        # coarse action discretization
            'step_duration': 1.0    # hours per step (1 hour)
        }
    ]
}

#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=mrdp_agent_kwargs, render=False, display_progress=True)

# Run all episodes in parallel using MRDP

mrdp_episode_logs = run_episodes_parallel(
    Agent, selected_list, agent_kwargs=mrdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False
)

[INFO] Starting 60 episodes with max_workers=12


Episodes:  50%|█████     | 30/60 [05:58<15:09, 30.31s/it]  

[START] Episode 9
Sim Complete
[DONE]  Episode 9 (Elapsed: 122.59 sec)
[START] Episode 14
Sim Complete
[DONE]  Episode 14 (Elapsed: 122.06 sec)
[START] Episode 26
Sim Complete
[DONE]  Episode 26 (Elapsed: 0.20 sec)
[START] Episode 27


Episodes:  52%|█████▏    | 31/60 [06:09<11:52, 24.58s/it]

[START] Episode 2
Sim Complete
[DONE]  Episode 2 (Elapsed: 124.38 sec)
[START] Episode 17
Sim Complete
[DONE]  Episode 17 (Elapsed: 122.20 sec)
[START] Episode 28
[START] Episode 4
Sim Complete
[DONE]  Episode 4 (Elapsed: 123.66 sec)
[START] Episode 16
Sim Complete
[DONE]  Episode 16 (Elapsed: 123.82 sec)
[START] Episode 29
[START] Episode 10
Sim Complete
[DONE]  Episode 10 (Elapsed: 120.86 sec)
[START] Episode 12
Sim Complete
[DONE]  Episode 12 (Elapsed: 1.26 sec)
[START] Episode 13
Sim Complete
[DONE]  Episode 13 (Elapsed: 122.01 sec)
[START] Episode 25


...

[START] Episode 6
Sim Complete
[DONE]  Episode 6 (Elapsed: 126.71 sec)
[START] Episode 23
Sim Complete
[DONE]  Episode 23 (Elapsed: 122.19 sec)
[START] Episode 31
Sim Complete
[DONE]  Episode 31 (Elapsed: 106.82 sec)
[START] Episode 41


Episodes:  98%|█████████▊| 59/60 [09:31<00:12, 12.51s/it]

Sim Complete
[DONE]  Episode 27 (Elapsed: 121.70 sec)
[START] Episode 42
Sim Complete
[DONE]  Episode 42 (Elapsed: 122.97 sec)
[START] Episode 57


Episodes: 100%|██████████| 60/60 [09:40<00:00,  9.67s/it]


Sim Complete
[DONE]  Episode 29 (Elapsed: 120.71 sec)
[START] Episode 44
Sim Complete
[DONE]  Episode 44 (Elapsed: 123.66 sec)
Sim Complete
[DONE]  Episode 28 (Elapsed: 121.40 sec)
[START] Episode 43
Sim Complete
[DONE]  Episode 43 (Elapsed: 120.29 sec)
[START] Episode 56
Sim Complete
[DONE]  Episode 56 (Elapsed: 2.16 sec)
[START] Episode 58
Sim Complete
[DONE]  Episode 58 (Elapsed: 1.55 sec)
Sim Complete
[DONE]  Episode 32 (Elapsed: 124.58 sec)
[START] Episode 47
Sim Complete
[DONE]  Episode 47 (Elapsed: 123.22 sec)
Sim Complete
[DONE]  Episode 34 (Elapsed: 124.67 sec)
[START] Episode 49
Sim Complete
[DONE]  Episode 49 (Elapsed: 123.67 sec)
Sim Complete
[DONE]  Episode 38 (Elapsed: 123.36 sec)
[START] Episode 50
Sim Complete
[DONE]  Episode 50 (Elapsed: 124.42 sec)
Sim Complete
[DONE]  Episode 36 (Elapsed: 126.31 sec)
[START] Episode 51
Sim Complete
[DONE]  Episode 51 (Elapsed: 121.32 sec)
Sim Complete
[DONE]  Episode 40 (Elapsed: 124.41 sec)
[START] Episode 53
Sim Complete
[DONE]  Ep

In [10]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(mrdp_episode_logs)]
mrdp_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/mrdp_{env_type}_{episode_num}_logs.parquet"
mrdp_episode_logs.write_parquet(file_name)

In [17]:
from decision import Agent, run_episodes_parallel

oracle_agent_kwargs = {
    'algorithm': 'oracle',
    # Optional: override horizon and action resolution for the oracle
    'soc_resolution': 20,
    'action_resolution': 41,  # number of discrete actions between -1 and 1
    'horizon': 48,  # plan 24 steps ahead (default: same as SDP horizon)

}

# Run all episodes in parallel using the oracle agent
oracle_episode_logs = run_episodes_parallel(
    Agent,
    selected_list[46:98],  # your list of environments
    agent_kwargs=oracle_agent_kwargs,
    max_workers=14,
    use_notebook_tqdm=False
)

[INFO] Starting 52 episodes with max_workers=14


Episodes: 100%|██████████| 52/52 [9:51:57<00:00, 683.03s/it]    


[START] Episode 10
Sim Complete
[DONE]  Episode 10 (Elapsed: 9750.13 sec)
[START] Episode 26
Sim Complete
[DONE]  Episode 26 (Elapsed: 9244.78 sec)
[START] Episode 40
Sim Complete
[DONE]  Episode 40 (Elapsed: 9030.67 sec)
[START] Episode 6
Sim Complete
[DONE]  Episode 6 (Elapsed: 9769.90 sec)
[START] Episode 27
Sim Complete
[DONE]  Episode 27 (Elapsed: 8925.21 sec)
[START] Episode 37
Sim Complete
[DONE]  Episode 37 (Elapsed: 9515.89 sec)
[START] Episode 5
Sim Complete
[DONE]  Episode 5 (Elapsed: 9451.21 sec)
[START] Episode 22
Sim Complete
[DONE]  Episode 22 (Elapsed: 9404.41 sec)
[START] Episode 38
Sim Complete
[DONE]  Episode 38 (Elapsed: 9493.17 sec)
[START] Episode 13
Sim Complete
[DONE]  Episode 13 (Elapsed: 9629.64 sec)
[START] Episode 24
Sim Complete
[DONE]  Episode 24 (Elapsed: 9415.94 sec)
[START] Episode 41
Sim Complete
[DONE]  Episode 41 (Elapsed: 9620.03 sec)
[START] Episode 3
Sim Complete
[DONE]  Episode 3 (Elapsed: 8900.08 sec)
[START] Episode 16
Sim Complete
[DONE]  Epis

In [18]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(oracle_episode_logs)]
oracle_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/oracle_{env_type}_{episode_num}_46-98_logs.parquet"
oracle_episode_logs.write_parquet(file_name)

In [ ]:
from decision import Agent, run_sb3_model_on_vec_env
from stable_baselines3 import PPO, A2C, DDPG, SAC, TD3
from helper import flatten_episode_data

# (only needed if you ever switch to SubprocVecEnv on Linux/notebooks)
multiprocessing.set_start_method("forkserver", force=True)

# Utility to yield batches from a list
def batchify(lst, batch_size):
    """Yield successive batches from lst of size batch_size."""
    for i in range(0, len(lst), batch_size):
        yield lst[i:i + batch_size]

from stable_baselines3.common.vec_env import SubprocVecEnv

batch_size = 64  # Set your desired batch size

In [ ]:
sac_model = SAC.load("../models/sac_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(sac_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

sac_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/sac_{env_type}_{episode_num}_logs.parquet"
sac_logs.write_parquet(file_name)

In [ ]:
PPO_model = PPO.load("../models/ppo_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(PPO_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

ppo_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/ppo_{env_type}_{episode_num}_logs.parquet"
ppo_logs.write_parquet(file_name)

In [ ]:
a2c_model = A2C.load("../models/a2c_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(a2c_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()

a2c_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/a2c_{env_type}_{episode_num}_logs.parquet"
a2c_logs.write_parquet(file_name)

In [ ]:
ddpg_model = DDPG.load("../models/ddpg_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(ddpg_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()
ddpg_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/ddpg_{env_type}_{episode_num}_logs.parquet"
ddpg_logs.write_parquet(file_name)

In [ ]:
td3_model = TD3.load("../models/td3_model.zip")
# Run test episodes in parallel in batches
all_episode_logs = []
for batch_num, env_fns_batch in enumerate(batchify(env_fns, batch_size)):
    print(f"Processing batch {batch_num+1}")
    vec_env = SubprocVecEnv(env_fns_batch)
    # Run your model on this batch
    episode_logs = run_sb3_model_on_vec_env(td3_model, vec_env)
    all_episode_logs.extend(episode_logs)
    vec_env.close()
td3_logs = flatten_episode_data(all_episode_logs)
file_name = f"../data/td3_{env_type}_{episode_num}_logs.parquet"
td3_logs.write_parquet(file_name)

In [ ]:
# to get the best RTG value, analysis of the distribution of total episode rewards in the dataset is needed


In [ ]:
from decision import Agent, run_episodes_parallel, run_single
from decision_transformer import DecisionTransformer
import json

import torch
# check if GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

with open('../models/decision_transformer_model_kwargs.json', 'r') as f:
    model_kwargs = json.load(f)

model = DecisionTransformer(**model_kwargs)

model.load_state_dict(torch.load('../models/dt_model.pt', map_location=device))
model.return_scale = 1.0  # or whatever was used during training
model.eval()
rtg = 4508964.69


dt_agent_kwargs = {
    'algorithm': 'dt',
    'model': model.to(device),
    'rtg_value': rtg
}

# check for nan in model parameters
import math
bad_params = [name for name, p in model.named_parameters() if torch.isnan(p).any() or torch.isinf(p).any()]
print("bad params:", bad_params)

In [ ]:
episode_log = run_episodes_parallel(Agent, test_envs, agent_kwargs=dt_agent_kwargs, max_workers=2, use_notebook_tqdm=False)

dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_log)]
dt_logs = pl.concat(dfs_with_id)
file_name = f"../data/dt_rtg{int(rtg)}_{env_type}_{episode_num}_logs.parquet"
dt_logs.write_parquet(file_name)